In [0]:
%sql
DROP TABLE IF EXISTS dim_date;

In [0]:
from pyspark.sql import functions as F

start_date = "2017-07-01" #first order
end_date   = "2021-12-31" #last target

date_gold = spark.sql(f"""SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS date""")

dim_date = date_gold.select(
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("date").alias("date"),
    F.year("date").alias("year"),
    F.month("date").alias("month"),
    F.dayofmonth("date").alias("day"),
    F.date_format("date", "yyyyMM").cast("int").alias("year_month"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.when(F.dayofweek("date").isin(1, 7), 1).otherwise(0).alias("IsWeekend"),
    F.trunc("date", "MM").alias("month_start_date") 
    #transforming the date to the first day of the month(15/05/2023 -> 01/05/2023) so we can join with fact_sales
)

(dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") #if we want to change any column later
    .saveAsTable("sales_lakehouse.gold.dim_date"))


In [0]:
dim_date.printSchema()

In [0]:
display(dim_date)